# Langbridge SDK Federated Query Notebook

This walkthrough shows the local Langbridge runtime federating three live sources at query time:

- a Postgres sales database
- a Postgres CRM database
- a local CSV with campaign attribution tags

The shared key is `contact_external_id`, so the runtime can combine revenue, CRM context, and marketing campaign data without copying them into one warehouse first.


In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

EXAMPLE_DIR = Path.cwd().resolve()
if not (EXAMPLE_DIR / "langbridge_config.yml").exists():
    candidate = EXAMPLE_DIR / "examples" / "sdk" / "federated_query"
    if candidate.exists():
        EXAMPLE_DIR = candidate.resolve()

REPO_ROOT = EXAMPLE_DIR.parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from langbridge import LangbridgeClient


## Start the Postgres sources

This example uses Docker to start the sales and CRM databases from the seed scripts in this folder.


In [ ]:
subprocess.run(["docker", "compose", "up", "-d", "--wait"], cwd=EXAMPLE_DIR, check=True)
subprocess.run(["docker", "compose", "ps"], cwd=EXAMPLE_DIR, check=True)


In [2]:
client = LangbridgeClient.local(config_path=str(EXAMPLE_DIR / "langbridge_config.yml"))
datasets = client.datasets.list()
dataset_ids = {item.name: item.id for item in datasets.items}

datasets_df = pd.DataFrame([item.model_dump(mode="json") for item in datasets.items])
datasets_df = datasets_df.sort_values("name").reset_index(drop=True)
datasets_df


,id,name,label,description,connector,semantic_model,managed
0,12d9bcee-b0bc-5689-afc1-cf150fab64b3,crm_contacts,CRM Contacts,CRM contacts enriched with account segment and...,crm_reporting,customer_revenue_federation,False
1,e367228d-1980-5ff1-8623-d63f9938d24a,marketing_campaigns,Marketing Campaigns,Campaign metadata loaded from a local CSV and ...,campaign_file,customer_revenue_federation,False
2,60ff3235-1b12-501d-a9bc-43ccc84129a0,sales_orders,Sales Orders,Order-level sales facts enriched with the CRM ...,sales_reporting,customer_revenue_federation,False


In [3]:
def preview_dataset(name: str, limit: int = 5) -> pd.DataFrame:
    result = client.datasets.query(dataset_id=dataset_ids[name], limit=limit)
    return pd.DataFrame(result.rows)

for dataset_name in ["sales_orders", "crm_contacts", "marketing_campaigns"]:
    print(dataset_name)
    display(preview_dataset(dataset_name))


sales_orders


/home/callumwhi/langbridgedev/langbridge/langbridge-connectors/langbridge-connector-snowflake/src/langbridge_connector_snowflake/config.py:11: UserWarning: Field name "schema" in "SnowflakeConnectorConfig" shadows an attribute in parent "BaseConnectorConfig"
  class SnowflakeConnectorConfig(BaseConnectorConfig):


,order_id,crm_contact_external_id,country,loyalty_tier,channel,status,order_date,order_total
0,1,CRM-00000001,US,bronze,store,returned,2026-03-06,794.83
1,2,CRM-00000001,US,bronze,store,returned,2026-03-06,3005.47
2,3,CRM-00000002,US,bronze,store,completed,2025-09-03,1860.36
3,4,CRM-00000002,US,bronze,store,completed,2026-03-12,472.38
4,5,CRM-00000003,US,bronze,online,processing,2025-11-11,1094.84


crm_contacts


,contact_external_id,account_segment,owner_region,lifecycle_stage,lead_score,preferred_channel,marketing_opt_in,last_touch_date
0,CRM-00000001,startup,West,prospect,95,email,True,2025-12-14
1,CRM-00000002,smb,North,prospect,51,email,True,2025-12-14
2,CRM-00000003,enterprise,Metro,customer,53,phone,True,2026-03-02
3,CRM-00000004,mid_market,Coastal,customer,19,chat,True,2026-03-13
4,CRM-00000005,smb,Enterprise,customer,26,chat,True,2026-03-01


marketing_campaigns


,contact_external_id,campaign_name,campaign_channel,offer_type,target_region,touch_date,engagement_score
0,CRM-00000001,Spring Refresh,email,discount,East,2026-01-05,91
1,CRM-00000002,Spring Refresh,email,discount,East,2026-01-06,87
2,CRM-00000003,Spring Refresh,email,discount,East,2026-01-06,84
3,CRM-00000004,Spring Refresh,email,discount,East,2026-01-07,79
4,CRM-00000005,Spring Refresh,email,discount,East,2026-01-08,88


## Federated semantic query

This semantic query asks for revenue from the sales database, grouped by CRM account segment and CSV campaign name.


In [4]:
semantic_result = client.semantic.query(
    "customer_revenue_federation",
    measures=["sales_orders.net_revenue", "sales_orders.order_count"],
    dimensions=["marketing_campaigns.campaign_name", "crm_contacts.account_segment"],
    filters=[{"member": "marketing_campaigns.campaign_name", "operator": "set"}],
    order={"sales_orders.net_revenue": "desc"},
    limit=12,
)

pd.DataFrame(semantic_result.rows)


,marketing_campaigns.campaign_name,crm_contacts.account_segment,sales_orders.net_revenue,sales_orders.order_count
0,Premium Upgrade,smb,25205.05,18
1,Summer Bundle,mid_market,17804.21,14
2,Reactivation Push,mid_market,17288.39,14
3,Reactivation Push,startup,17067.88,12
4,Spring Refresh,smb,16504.22,12
5,Store Launch,smb,16457.33,12
6,VIP Retention,enterprise,14640.71,11
7,VIP Retention,smb,14604.57,12
8,Store Launch,startup,14091.88,10
9,Summer Bundle,startup,13756.06,9


## Direct federated SQL

Here the runtime federates across the eligible workspace datasets and derives the logical SQL aliases from dataset metadata.


In [6]:
sql_result = client.sql.query(
    query="""
    SELECT
      m.campaign_name,
      c.account_segment,
      COUNT(DISTINCT s.order_id) AS order_count,
      ROUND(SUM(s.order_total), 2) AS net_revenue
    FROM sales_orders AS s
    JOIN crm_contacts AS c
      ON s.crm_contact_external_id = c.contact_external_id
    JOIN marketing_campaigns AS m
      ON c.contact_external_id = m.contact_external_id
    GROUP BY m.campaign_name, c.account_segment
    ORDER BY net_revenue DESC, order_count DESC
    LIMIT 5
    """,
)

pd.DataFrame(sql_result.rows)


,campaign_name,account_segment,order_count,net_revenue
0,Premium Upgrade,smb,18,25205.05
1,Summer Bundle,mid_market,14,17804.21
2,Reactivation Push,mid_market,14,17288.39
3,Reactivation Push,startup,12,17067.88
4,Spring Refresh,smb,12,16504.22


When you are done, run `docker compose down -v` from this folder to remove the demo databases.
